# LangChain Complete Guide
## RAG | Agents | LangGraph | LangSmith

This notebook covers:
1. RAG Without LangChain (manual implementation)
2. RAG With LangChain (using LangChain components)
3. LangGraph Agent with Tools, Guardrails, and Orchestrator
4. LangSmith Tracing

Model used throughout: `gpt-4o-mini`

---
## SECTION 0 - Installation and Setup

In [1]:
import subprocess, sys

packages = [
    "openai", "langchain", "langchain-openai", "langchain-community",
    "langchain-core", "langgraph", "langsmith",
    "faiss-cpu", "tiktoken", "numpy", "requests", "python-dotenv", "chromadb"
]

print("📦 Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)
print("✅ All dependencies installed successfully")

📦 Installing dependencies...
✅ All dependencies installed successfully



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

# ── API Keys ──────────────────────────────────────────────────────────────────
# Fill these in if not using a .env file
os.environ["OPENAI_API_KEY"]      = "sk-proj---_JkUyKXx2owDlixPWjuWFoNETNKToFO96ltaWLtL37XDDYevMA"
os.environ["LANGCHAIN_API_KEY"]   = ""
# os.environ["OPENWEATHER_API_KEY"] = "..."

os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGCHAIN_PROJECT", "langchanin-graph")

openai_key    = os.environ.get("OPENAI_API_KEY", "")
langsmith_key = os.environ.get("LANGCHAIN_API_KEY", "")

print("[SETUP] Environment variables configured.")
print(f"[SETUP] OpenAI key   : {'✅ set' if openai_key.startswith('sk-') else '❌ missing'}")
print(f"[SETUP] LangSmith key: {'✅ set' if langsmith_key.startswith('lsv2') else '⚠️  not set (tracing disabled)'}")

[SETUP] Environment variables configured.
[SETUP] OpenAI key   : ✅ set
[SETUP] LangSmith key: ✅ set


In [15]:
from openai import OpenAI

client = OpenAI()   # reads OPENAI_API_KEY from env
print("✅ OpenAI client initialized")

✅ OpenAI client initialized


---
## SECTION 1 - RAG Without LangChain

Every component is built manually:
- **Chunking** – character-level sliding window
- **Embeddings** – direct OpenAI API call
- **Vector store** – in-memory cosine similarity
- **Retrieval + Generation** – full RAG pipeline

In [16]:
# ── Document ──────────────────────────────────────────────────────────────────
RAW_DOCUMENT = """
LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).
It provides tools for chaining prompts, managing memory, integrating with external data sources, and building agents.

LangGraph is a library built on top of LangChain that enables building stateful, multi-actor applications with LLMs.
It uses a graph-based architecture where nodes represent computation steps and edges represent transitions.
LangGraph is particularly useful for building complex agent workflows that require cycles and branching logic.

LangSmith is a developer platform for debugging, testing, evaluating, and monitoring LLM applications.
It provides tracing capabilities that allow developers to see every step of an LLM chain or agent run.
LangSmith integrates seamlessly with LangChain and LangGraph.

RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrieved
from a knowledge base and provided to the LLM as context before generating a response.
RAG helps reduce hallucinations and allows the model to answer questions about private or recent data.

Embeddings are numerical vector representations of text. Similar texts produce similar embedding vectors.
This property allows us to find semantically similar documents using distance metrics like cosine similarity.
OpenAI's text-embedding-ada-002 and text-embedding-3-small are commonly used embedding models.

Vector stores are databases optimized for storing and searching embedding vectors.
FAISS (Facebook AI Similarity Search) is a popular open-source vector store for fast nearest-neighbor search.
Other vector stores include Chroma, Pinecone, Weaviate, and Qdrant.

Agents are LLM-powered systems that can use tools to accomplish tasks.
A tool is a function that the agent can call, such as a web search, calculator, or API call.
Agents decide which tools to use and in what order based on the user's query.
"""

print(f"[DOCUMENT] {len(RAW_DOCUMENT)} chars | {len(RAW_DOCUMENT.split())} words")

[DOCUMENT] 1948 chars | 278 words


In [ ]:
# ── Optional: load from PDF instead ──────────────────────────────────────────
# from langchain_community.document_loaders import PyPDFLoader
#
# loader = PyPDFLoader("your_file.pdf")
# pdf_docs = loader.load()
# RAW_DOCUMENT = "\n\n".join(doc.page_content.strip() for doc in pdf_docs if doc.page_content.strip())
# print(f"[PDF] {len(pdf_docs)} pages | {len(RAW_DOCUMENT)} chars")

In [5]:
# ── Chunking ──────────────────────────────────────────────────────────────────
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> list[str]:
    """Split text into overlapping character-level chunks by paragraph."""
    paragraphs = [p.strip() for p in text.strip().split("\n\n") if p.strip()]
    chunks = []
    for para in paragraphs:
        start = 0
        while start < len(para):
            chunk = para[start : start + chunk_size].strip()
            if chunk:
                chunks.append(chunk)
            start += chunk_size - overlap
    return chunks


chunks = chunk_text(RAW_DOCUMENT, chunk_size=250, overlap=50)
print(f"[CHUNKING] {len(chunks)} chunks produced")
for i, chunk in enumerate(chunks, 1):
    print(f"  Chunk {i:02d} ({len(chunk)} chars): {chunk[:80]}...")

[CHUNKING] 14 chunks produced
  Chunk 01 (228 chars): LangChain is a framework designed to simplify the creation of applications using...
  Chunk 02 (28 chars): ources, and building agents....
  Chunk 03 (250 chars): LangGraph is a library built on top of LangChain that enables building stateful,...
  Chunk 04 (135 chars): s represent transitions.
LangGraph is particularly useful for building complex a...
  Chunk 05 (250 chars): LangSmith is a developer platform for debugging, testing, evaluating, and monito...
  Chunk 06 (66 chars): run.
LangSmith integrates seamlessly with LangChain and LangGraph....
  Chunk 07 (250 chars): RAG stands for Retrieval-Augmented Generation. It is a technique where relevant ...
  Chunk 08 (92 chars): reduce hallucinations and allows the model to answer questions about private or ...
  Chunk 09 (250 chars): Embeddings are numerical vector representations of text. Similar texts produce s...
  Chunk 10 (110 chars): ine similarity.
OpenAI's text-embedding-ada

In [6]:
# ── Embeddings ────────────────────────────────────────────────────────────────
import numpy as np

def get_embeddings(texts: list[str], model: str = "text-embedding-3-small") -> list:
    """Embed texts in batches of 2000 to stay within API limits."""
    all_vectors = []
    for i in range(0, len(texts), 2000):
        batch = texts[i : i + 2000]
        response = client.embeddings.create(input=batch, model=model)
        all_vectors.extend(item.embedding for item in response.data)
    print(f"[EMBEDDINGS] {len(all_vectors)} vectors | dim={len(all_vectors[0])}")
    return all_vectors


chunk_embeddings = get_embeddings(chunks)

[EMBEDDINGS] 14 vectors | dim=1536


In [ ]:
# ── In-Memory Vector Store ────────────────────────────────────────────────────
class SimpleVectorStore:
    """Minimal in-memory vector store using cosine similarity."""

    def __init__(self):
        self.texts: list[str] = []
        self.embeddings: list[np.ndarray] = []

    def add(self, texts: list[str], embeddings: list) -> None:
        self.texts.extend(texts)
        self.embeddings.extend(np.array(e) for e in embeddings)
        print(f"[VECTOR STORE] {len(self.texts)} documents indexed")

    @staticmethod
    def _cosine(a: np.ndarray, b: np.ndarray) -> float:
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

    def search(self, query_embedding: list, top_k: int = 3) -> list[tuple[float, str]]:
        q = np.array(query_embedding)
        scored = sorted(
            ((self._cosine(q, e), t) for e, t in zip(self.embeddings, self.texts)),
            reverse=True
        )
        return scored[:top_k]


vector_store = SimpleVectorStore()
vector_store.add(chunks, chunk_embeddings)

In [ ]:
# ── RAG Pipeline ─────────────────────────────────────────────────────────────
def rag_query_no_langchain(question: str, top_k: int = 3) -> str:
    """End-to-end RAG: embed → retrieve → generate."""
    print(f"\n{'='*60}\n[RAG] {question}\n{'='*60}")

    query_vec = get_embeddings([question])[0]
    results   = vector_store.search(query_vec, top_k=top_k)

    print(f"[RETRIEVE] Top {top_k} chunks:")
    for rank, (score, text) in enumerate(results, 1):
        print(f"  {rank}. score={score:.4f} | {text[:100]}...")

    context = "\n\n".join(text for _, text in results)
    prompt = (
        "You are a helpful assistant. Use ONLY the context below to answer.\n"
        "If the answer is not in the context, say \"I don't know based on the provided context.\"\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a concise and accurate assistant."},
            {"role": "user",   "content": prompt}
        ],
        temperature=0
    )
    answer = response.choices[0].message.content
    print(f"\n[ANSWER]\n{answer}")
    return answer


rag_query_no_langchain("What is RAG and why is it useful?")

In [ ]:
rag_query_no_langchain("What is LangGraph and how does it differ from LangChain?")

---
## SECTION 2 - RAG With LangChain

Same pipeline rebuilt with LangChain components:
`Document → RecursiveCharacterTextSplitter → OpenAIEmbeddings → Chroma → Retriever → LCEL Chain`

In [17]:
# ── Document Loading & Chunking ──────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

raw_doc = Document(
    page_content=RAW_DOCUMENT,
    metadata={"source": "manual", "title": "LangChain Overview"}
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)
lc_chunks = splitter.split_documents([raw_doc])

print(f"[CHUNKING] {len(lc_chunks)} chunks")
for i, chunk in enumerate(lc_chunks, 1):
    print(f"  Chunk {i:02d} ({len(chunk.page_content)} chars): {chunk.page_content[:80]}...")

[CHUNKING] 12 chunks
  Chunk 01 (228 chars): LangChain is a framework designed to simplify the creation of applications using...
  Chunk 02 (224 chars): LangGraph is a library built on top of LangChain that enables building stateful,...
  Chunk 03 (110 chars): LangGraph is particularly useful for building complex agent workflows that requi...
  Chunk 04 (205 chars): LangSmith is a developer platform for debugging, testing, evaluating, and monito...
  Chunk 05 (61 chars): LangSmith integrates seamlessly with LangChain and LangGraph....
  Chunk 06 (190 chars): RAG stands for Retrieval-Augmented Generation. It is a technique where relevant ...
  Chunk 07 (102 chars): RAG helps reduce hallucinations and allows the model to answer questions about p...
  Chunk 08 (215 chars): Embeddings are numerical vector representations of text. Similar texts produce s...
  Chunk 09 (94 chars): OpenAI's text-embedding-ada-002 and text-embedding-3-small are commonly used emb...
  Chunk 10 (192 chars): Vect

In [18]:
# ── Embeddings ────────────────────────────────────────────────────────────────
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.environ["OPENAI_API_KEY"]
)

sample_vec = embeddings_model.embed_query("What is retrieval augmented generation?")
print(f"[EMBEDDINGS] model=text-embedding-3-small | dim={len(sample_vec)}")

[EMBEDDINGS] model=text-embedding-3-small | dim=1536


In [19]:
# ── Vector Store (Chroma) ─────────────────────────────────────────────────────
from langchain_community.vectorstores import Chroma

chroma_store = Chroma.from_documents(
    documents=lc_chunks,
    embedding=embeddings_model,
    persist_directory="./chroma_db"
)
print(f"[VECTOR STORE] Chroma built | {len(lc_chunks)} documents indexed")

# Sanity check
results = chroma_store.similarity_search_with_score("vector stores and FAISS", k=3)
print("\n[VECTOR STORE] Test search — top 3:")
for rank, (doc, score) in enumerate(results, 1):
    print(f"  {rank}. score={score:.4f} | {doc.page_content[:80]}...")

[VECTOR STORE] Chroma built | 12 documents indexed

[VECTOR STORE] Test search — top 3:
  1. score=0.5924 | Vector stores are databases optimized for storing and searching embedding vector...
  2. score=0.5924 | Vector stores are databases optimized for storing and searching embedding vector...
  3. score=0.9508 | Other vector stores include Chroma, Pinecone, Weaviate, and Qdrant....


In [20]:
# ── Retriever ─────────────────────────────────────────────────────────────────
retriever = chroma_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

test_docs = retriever.invoke("How do agents use tools?")
print(f"[RETRIEVER] {len(test_docs)} docs retrieved")
for i, doc in enumerate(test_docs, 1):
    print(f"  Doc {i}: {doc.page_content[:100]}...")

[RETRIEVER] 3 docs retrieved
  Doc 1: Agents are LLM-powered systems that can use tools to accomplish tasks.
A tool is a function that the...
  Doc 2: Agents are LLM-powered systems that can use tools to accomplish tasks.
A tool is a function that the...
  Doc 3: LangGraph is particularly useful for building complex agent workflows that require cycles and branch...


In [21]:
# ── LCEL RAG Chain ────────────────────────────────────────────────────────────
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.environ["OPENAI_API_KEY"]
)

rag_prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant.

Answer the question using ONLY the provided context.
If the answer is not there, respond with: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
print("[CHAIN] Retriever → Prompt → LLM → StrOutputParser — assembled")

[CHAIN] Retriever → Prompt → LLM → StrOutputParser — assembled


In [22]:
# ── Run Queries ───────────────────────────────────────────────────────────────
def rag_query_langchain(question: str) -> str:
    print(f"\n{'='*60}\n[RAG-LC] {question}\n{'='*60}")
    retrieved = retriever.invoke(question)
    print(f"[RETRIEVE] {len(retrieved)} chunks:")
    for i, doc in enumerate(retrieved, 1):
        print(f"  {i}. {doc.page_content[:100]}...")
    answer = rag_chain.invoke(question)
    print(f"\n[ANSWER]\n{answer}")
    return answer


rag_query_langchain("What is LangSmith used for?")


[RAG-LC] What is LangSmith used for?
[RETRIEVE] 3 chunks:
  1. LangSmith integrates seamlessly with LangChain and LangGraph....
  2. LangSmith integrates seamlessly with LangChain and LangGraph....
  3. LangSmith is a developer platform for debugging, testing, evaluating, and monitoring LLM application...

[ANSWER]
LangSmith is used for debugging, testing, evaluating, and monitoring LLM applications.


'LangSmith is used for debugging, testing, evaluating, and monitoring LLM applications.'

In [23]:
rag_query_langchain("What is IIMA?")


[RAG-LC] What is IIMA?
[RETRIEVE] 3 chunks:
  1. LangSmith is a developer platform for debugging, testing, evaluating, and monitoring LLM application...
  2. LangSmith is a developer platform for debugging, testing, evaluating, and monitoring LLM application...
  3. RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrie...

[ANSWER]
I don't know based on the provided context.


"I don't know based on the provided context."

---
## SECTION 3 - LangGraph Agent

Stateful agent with:
- **Tools**: `get_weather`, `calculator`, `get_definition`
- **Guardrail**: keyword-based input filter
- **Graph**: `llm ↔ tools` loop with conditional routing

In [ ]:
# ── Tool Definitions ──────────────────────────────────────────────────────────
import math
import requests
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get current weather for a city (temperature °C, condition, humidity)."""
    print(f"[TOOL: get_weather] city='{city}'")
    api_key = os.environ.get("OPENWEATHER_API_KEY", "")
    if api_key:
        url = (
            f"http://api.openweathermap.org/data/2.5/weather"
            f"?q={city}&appid={api_key}&units=metric"
        )
        try:
            resp = requests.get(url, timeout=5)
            if resp.status_code == 200:
                d = resp.json()
                result = (
                    f"Weather in {city}: {d['main']['temp']}C, "
                    f"{d['weather'][0]['description']}, "
                    f"humidity {d['main']['humidity']}%"
                )
                print(f"[TOOL: get_weather] {result}")
                return result
        except Exception as e:
            print(f"[TOOL: get_weather] API error: {e} — falling back to mock")

    mock = {
        "london":    "Weather in London: 12C, overcast clouds, humidity 78%",
        "new york":  "Weather in New York: 18C, clear sky, humidity 55%",
        "tokyo":     "Weather in Tokyo: 22C, light rain, humidity 82%",
        "paris":     "Weather in Paris: 15C, partly cloudy, humidity 65%",
        "hyderabad": "Weather in Hyderabad: 35C, sunny, humidity 40%",
    }
    result = mock.get(city.lower(), f"Weather in {city}: 20C, clear sky, humidity 60% (mock)")
    print(f"[TOOL: get_weather] {result}")
    return result


@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression. Examples: '2+2', 'sqrt(16)', '100*1.08'."""
    print(f"[TOOL: calculator] expr='{expression}'")
    try:
        safe_ns = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
        safe_ns["abs"] = abs
        result = eval(expression, {"__builtins__": {}}, safe_ns)
        output = f"Result of '{expression}' = {result}"
        print(f"[TOOL: calculator] {output}")
        return output
    except Exception as e:
        error = f"Could not evaluate '{expression}': {e}"
        print(f"[TOOL: calculator] ERROR: {error}")
        return error


@tool
def get_definition(term: str) -> str:
    """Return a short definition of a technical term."""
    print(f"[TOOL: get_definition] term='{term}'")
    definitions = {
        "rag":          "RAG (Retrieval-Augmented Generation) combines document retrieval with LLM generation to answer questions using external knowledge.",
        "langchain":    "LangChain is a Python/TypeScript framework for building LLM-powered applications with chains, agents, and memory.",
        "langgraph":    "LangGraph is a LangChain extension for building stateful multi-actor workflows using a directed graph structure.",
        "embedding":    "An embedding is a high-dimensional numerical vector representing the semantic meaning of a piece of text.",
        "faiss":        "FAISS (Facebook AI Similarity Search) is an open-source library for efficient similarity search over dense vectors.",
        "agent":        "An agent is an LLM-powered system that autonomously chooses and calls tools to complete a task.",
        "vector store": "A vector store is a database optimized for storing and retrieving high-dimensional embedding vectors via similarity search.",
    }
    result = definitions.get(
        term.lower().strip(),
        f"No definition found for '{term}'. Known terms: rag, langchain, langgraph, embedding, faiss, agent, vector store"
    )
    print(f"[TOOL: get_definition] {result[:80]}...")
    return result


print("[TOOLS] Registered: get_weather | calculator | get_definition")

In [ ]:
# ── Guardrail ─────────────────────────────────────────────────────────────────
BLOCKED_KEYWORDS = ["bomb", "weapon", "hack", "exploit", "malware", "virus", "illegal", "drugs", "nsfw"]

def guardrail_check(user_input: str) -> tuple[bool, str]:
    """Returns (is_safe, reason). Blocks short inputs and harmful keywords."""
    print(f"[GUARDRAIL] Checking: '{user_input[:80]}'")
    if len(user_input.strip()) < 3:
        print("[GUARDRAIL] BLOCKED — too short")
        return False, "Input is too short. Please provide a meaningful question."
    for kw in BLOCKED_KEYWORDS:
        if kw in user_input.lower():
            print(f"[GUARDRAIL] BLOCKED — keyword='{kw}'")
            return False, f"Disallowed content detected: '{kw}'."
    print("[GUARDRAIL] PASSED")
    return True, "OK"

In [ ]:
# ── LangGraph Agent ───────────────────────────────────────────────────────────
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
import operator

tools = [get_weather, calculator, get_definition]

agent_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=os.environ["OPENAI_API_KEY"]
).bind_tools(tools)

AGENT_SYSTEM = SystemMessage(content=(
    "You are a helpful assistant with access to weather, calculator, and definition tools. "
    "Use tools when needed to answer accurately."
))


class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]


def call_llm(state: AgentState) -> AgentState:
    print(f"\n[LLM] Invoking with {len(state['messages'])} message(s)...")
    response = agent_llm.invoke([AGENT_SYSTEM] + state["messages"])
    if response.tool_calls:
        print(f"[LLM] Tool calls: {[tc['name'] for tc in response.tool_calls]}")
    else:
        print(f"[LLM] Final answer: {response.content[:80]}...")
    return {"messages": [response]}


def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        print("[ROUTER] → tools")
        return "tools"
    print("[ROUTER] → END")
    return END


tool_node = ToolNode(tools)

workflow = StateGraph(AgentState)
workflow.add_node("llm", call_llm)
workflow.add_node("tools", tool_node)
workflow.set_entry_point("llm")
workflow.add_conditional_edges("llm", should_continue)
workflow.add_edge("tools", "llm")

agent_app = workflow.compile()
print("[AGENT] Graph compiled: llm ↔ tools loop")

In [ ]:
# ── Visualize Graph ───────────────────────────────────────────────────────────
from IPython.display import Image, display, Markdown

def show_langgraph(app, title: str):
    display(Markdown(f"### {title}"))
    graph = app.get_graph()
    try:
        display(Image(graph.draw_mermaid_png()))
    except Exception as e:
        print(f"PNG render failed ({e}). Mermaid source:\n")
        print(graph.draw_mermaid())

show_langgraph(agent_app, "LangGraph Agent (llm ↔ tools)")

In [ ]:
# ── Run Agent ─────────────────────────────────────────────────────────────────
def run_agent(user_input: str) -> str:
    print(f"\n{'='*60}\n[AGENT] {user_input}\n{'='*60}")
    is_safe, reason = guardrail_check(user_input)
    if not is_safe:
        print(f"[BLOCKED] {reason}")
        return reason
    final_state = agent_app.invoke({"messages": [HumanMessage(content=user_input)]})
    answer = final_state["messages"][-1].content
    print(f"\n[ANSWER] {answer}")
    return answer


run_agent("What is the weather like in Sydney right now?")

In [ ]:
run_agent("What is 45 / 8500, then multiply that by 3?")

In [ ]:
run_agent("What is an Aeroplane? Give me a short definition.")

In [ ]:
# Guardrail test
run_agent("How do I hack into a system?")

---
## SECTION 4 - Orchestrator Agent

An LLM-based router dispatches queries to the right specialist:
- **weather_agent** – weather lookups
- **math_agent** – arithmetic / calculations
- **knowledge_agent** – definitions / explanations
- **general_agent** – fallback

In [ ]:
# ── Sub-Agents ────────────────────────────────────────────────────────────────
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

base_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=os.environ["OPENAI_API_KEY"]
)


def _llm(system: str, user: str) -> str:
    """Single-turn LLM helper."""
    return base_llm.invoke([SystemMessage(content=system), HumanMessage(content=user)]).content


def weather_agent(query: str) -> str:
    print("[SUB-AGENT: weather]")
    city = _llm("Extract only the city name. Reply with just the city name.", query).strip()
    print(f"  City: '{city}'")
    weather = get_weather.invoke({"city": city})
    return _llm("You are a friendly weather reporter. Summarize in one sentence.", f"Data: {weather}")


def math_agent(query: str) -> str:
    print("[SUB-AGENT: math]")
    expr = _llm("Extract a Python-evaluable math expression. Reply with ONLY the expression.", query).strip()
    print(f"  Expression: '{expr}'")
    result = calculator.invoke({"expression": expr})
    return _llm("You are a math tutor. Explain the result in one sentence.", f"Q: {query}\nCalc: {result}")


def knowledge_agent(query: str) -> str:
    print("[SUB-AGENT: knowledge]")
    term = _llm("Extract the key technical term the user wants defined. Reply with only that term.", query).strip()
    print(f"  Term: '{term}'")
    return get_definition.invoke({"term": term})


def general_agent(query: str) -> str:
    print("[SUB-AGENT: general]")
    return _llm("You are a helpful assistant. Answer concisely.", query)


print("[ORCHESTRATOR] Sub-agents ready: weather | math | knowledge | general")

In [ ]:
# ── Orchestrator Graph ────────────────────────────────────────────────────────
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator


class OrchestratorState(TypedDict):
    query: str
    route: str
    result: str
    guardrail_passed: bool
    log: Annotated[list, operator.add]


def guardrail_node(state: OrchestratorState) -> OrchestratorState:
    is_safe, reason = guardrail_check(state["query"])
    if not is_safe:
        return {"guardrail_passed": False, "result": reason, "log": [f"BLOCKED: {reason}"]}
    return {"guardrail_passed": True, "log": ["GUARDRAIL PASSED"]}


def router_node(state: OrchestratorState) -> OrchestratorState:
    route = _llm(
        "Classify the query as exactly one of: weather, math, knowledge, general.\n"
        "- weather: current weather / temperature\n"
        "- math: calculations or arithmetic\n"
        "- knowledge: definitions or technical explanations\n"
        "- general: anything else\n"
        "Reply with only the category name, lowercase.",
        state["query"]
    ).strip().lower()
    if route not in ("weather", "math", "knowledge", "general"):
        route = "general"
    print(f"[ROUTER] → {route}")
    return {"route": route, "log": [f"ROUTED TO: {route}"]}


def _make_agent_node(agent_fn):
    def node(state: OrchestratorState) -> OrchestratorState:
        result = agent_fn(state["query"])
        return {"result": result, "log": [f"result: {result[:60]}"]}
    return node


orchestrator = StateGraph(OrchestratorState)
orchestrator.add_node("guardrail",       guardrail_node)
orchestrator.add_node("router",          router_node)
orchestrator.add_node("weather_agent",   _make_agent_node(weather_agent))
orchestrator.add_node("math_agent",      _make_agent_node(math_agent))
orchestrator.add_node("knowledge_agent", _make_agent_node(knowledge_agent))
orchestrator.add_node("general_agent",   _make_agent_node(general_agent))

orchestrator.set_entry_point("guardrail")
orchestrator.add_conditional_edges(
    "guardrail",
    lambda s: "router" if s["guardrail_passed"] else END
)
orchestrator.add_conditional_edges(
    "router",
    lambda s: {"weather": "weather_agent", "math": "math_agent",
               "knowledge": "knowledge_agent", "general": "general_agent"}[s["route"]]
)
for node in ["weather_agent", "math_agent", "knowledge_agent", "general_agent"]:
    orchestrator.add_edge(node, END)

orchestrator_app = orchestrator.compile()
print("[ORCHESTRATOR] Graph compiled: guardrail → router → (specialist) → END")

In [ ]:
# ── Run Orchestrator ──────────────────────────────────────────────────────────
def run_orchestrator(query: str) -> str:
    print(f"\n{'='*60}\n[ORCHESTRATOR] {query}\n{'='*60}")
    initial = {"query": query, "route": "", "result": "", "guardrail_passed": False, "log": []}
    final = orchestrator_app.invoke(initial)
    print(f"\n[LOG] {' | '.join(final['log'])}")
    print(f"\n[RESULT]\n{final['result']}")
    return final["result"]


run_orchestrator("What is the weather in Tokyo?")

In [ ]:
run_orchestrator("Calculate the square root of 144 plus 50")

In [ ]:
run_orchestrator("What is LangGraph?")

In [ ]:
run_orchestrator("Tell me a fun fact about the ocean")

In [ ]:
# Guardrail test
run_orchestrator("How do I make a virus?")

In [ ]:
show_langgraph(orchestrator_app, "LangGraph Orchestrator")

---
## SECTION 5 - LangSmith Tracing

LangSmith auto-traces all LangChain calls when `LANGCHAIN_TRACING_V2=true`.
Below we also add **explicit named spans** via `@traceable` and list recent runs programmatically.

In [ ]:
# ── Setup & Verify ────────────────────────────────────────────────────────────
from langsmith import Client as LangSmithClient, traceable

ls_api_key = os.environ.get("LANGCHAIN_API_KEY", "")
ls_client  = LangSmithClient(api_key=ls_api_key) if ls_api_key.startswith("lsv2") else None

if ls_client:
    project = os.environ.get("LANGCHAIN_PROJECT", "default")
    print("[LANGSMITH] ✅ Client initialized")
    print(f"[LANGSMITH] Project : {project}")
    print(f"[LANGSMITH] Tracing : {os.environ.get('LANGCHAIN_TRACING_V2', 'false')}")
    print("[LANGSMITH] All LangChain calls above are already traced.")
    print("[LANGSMITH] View at : https://smith.langchain.com")
else:
    print("[LANGSMITH] ⚠️  No valid key — tracing disabled.")
    print("[LANGSMITH] Set LANGCHAIN_API_KEY (lsv2_...) to enable.")

In [ ]:
# ── Explicit @traceable Spans ─────────────────────────────────────────────────
@traceable(name="TracedRAGQuery", run_type="chain")
def traced_rag_query(question: str) -> dict:
    """RAG pipeline with an explicit LangSmith trace span."""
    retrieved = retriever.invoke(question)
    answer    = rag_chain.invoke(question)
    print(f"[TRACED-RAG] docs={len(retrieved)} | answer={answer[:80]}...")
    return {"question": question, "num_retrieved_docs": len(retrieved), "answer": answer}


@traceable(name="TracedAgentRun", run_type="chain")
def traced_agent_run(user_input: str) -> dict:
    """LangGraph agent run with an explicit LangSmith trace span."""
    is_safe, reason = guardrail_check(user_input)
    if not is_safe:
        return {"input": user_input, "blocked": True, "reason": reason}
    final_state = agent_app.invoke({"messages": [HumanMessage(content=user_input)]})
    answer = final_state["messages"][-1].content
    print(f"[TRACED-AGENT] answer={answer[:80]}...")
    return {"input": user_input, "blocked": False, "answer": answer}


print("[LANGSMITH] Traceable functions ready: traced_rag_query | traced_agent_run")

In [ ]:
# ── Run Traced RAG Query ──────────────────────────────────────────────────────
rag_result = traced_rag_query("What is the difference between LangChain and LangGraph?")
print("\n[RESULT]")
for k, v in rag_result.items():
    print(f"  {k}: {str(v)[:100]}")

In [ ]:
# ── Run Traced Agent ──────────────────────────────────────────────────────────
agent_result = traced_agent_run("What is the weather in Paris and what is 25 squared?")
print("\n[RESULT]")
for k, v in agent_result.items():
    print(f"  {k}: {str(v)[:100]}")

In [ ]:
# ── List Recent Runs ──────────────────────────────────────────────────────────
if ls_client:
    try:
        project = os.environ.get("LANGCHAIN_PROJECT", "default")
        runs = list(ls_client.list_runs(project_name=project, limit=10))
        if runs:
            print(f"[LANGSMITH] {len(runs)} recent runs in '{project}':\n")
            print(f"{'NAME':<35} {'STATUS':<12} {'LATENCY(s)':<14} {'TOKENS'}")
            print("-" * 75)
            for r in runs:
                name    = (r.name or "unnamed")[:32]
                status  = r.status or "unknown"
                latency = round(r.latency, 2) if r.latency else "N/A"
                tokens  = r.total_tokens if r.total_tokens else "N/A"
                print(f"{name:<35} {status:<12} {str(latency):<14} {tokens}")
        else:
            print("[LANGSMITH] No runs found yet.")
        print(f"\n[LANGSMITH] https://smith.langchain.com/o/your-org/projects/{project}")
    except Exception as e:
        print(f"[LANGSMITH] Could not fetch runs: {e}")
else:
    print("[LANGSMITH] Tracing not configured.")

---
## Summary

**Section 1 – RAG Without LangChain**
- Manual chunking, direct OpenAI Embeddings API, in-memory cosine-similarity store, GPT-4o-mini generation

**Section 2 – RAG With LangChain**
- `RecursiveCharacterTextSplitter` → `OpenAIEmbeddings` → `Chroma` → LCEL chain

**Section 3 – LangGraph Agent**
- `get_weather`, `calculator`, `get_definition` tools; keyword guardrail; `llm ↔ tools` loop

**Section 4 – Orchestrator Agent**
- LLM-based router → four specialist sub-agents; guardrail as entry node

**Section 5 – LangSmith Tracing**
- Automatic tracing via `LANGCHAIN_TRACING_V2=true`
- Explicit spans with `@traceable`; programmatic run listing via LangSmith client

---
*Model used throughout: `gpt-4o-mini`*